# 测试 PyTorch 模型与训练代码

## 学习目标

为模型前向、梯度更新、checkpoint 和可复现性编写快速测试，使维度或训练逻辑回归能够尽早暴露。

## 概念模型

测试不需要完整训练到高准确率。优先验证稳定契约：输入输出 shape、输出有限、参数能更新、保存加载一致，以及相同随机种子产生相同结果。

In [ ]:
import tempfile
from pathlib import Path
import torch
from torch import nn
from common.checkpoint import save_checkpoint, load_checkpoint
from common.runtime import seed_everything

class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
    def forward(self, inputs):
        return self.net(inputs)

model = TinyClassifier()
sample = torch.randn(5, 4)
print(model(sample).shape)

### 实验 1：前向契约测试

前向测试应覆盖正常 batch、不同 batch size 和明显错误的 feature 维度。

In [ ]:
output = model(sample)
assert output.shape == (5, 3)
assert torch.isfinite(output).all()
assert model(torch.randn(1, 4)).shape == (1, 3)
try:
    model(torch.randn(2, 5))
except RuntimeError as error:
    print('expected feature error:', type(error).__name__)
else:
    raise AssertionError('wrong feature dimension should fail')

### 实验 2：梯度和参数更新测试

只检查一次优化步骤是否改变参数，比等待完整训练更快地发现 detach、参数未注册或漏掉 `step()`。

In [ ]:
labels = torch.tensor([0, 1, 2, 0, 1])
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
before = {name: value.detach().clone() for name, value in model.named_parameters()}
optimizer.zero_grad(set_to_none=True)
loss = nn.CrossEntropyLoss()(model(sample), labels)
loss.backward()
assert all(parameter.grad is not None for parameter in model.parameters())
assert all(torch.isfinite(parameter.grad).all() for parameter in model.parameters())
optimizer.step()
changed = [not torch.equal(before[name], value.detach()) for name, value in model.named_parameters()]
assert any(changed)
print('loss and update:', loss.item(), changed)

### 实验 3：checkpoint round-trip 和随机种子

保存加载测试必须比较模型输出，而不是只检查文件存在。

In [ ]:
model.eval()
with torch.inference_mode():
    expected = model(sample)
with tempfile.TemporaryDirectory() as directory:
    path = save_checkpoint(Path(directory) / 'model.pt', model, optimizer, epoch=1, metrics={'loss': loss.item()})
    restored = TinyClassifier().eval()
    restored_optimizer = torch.optim.SGD(restored.parameters(), lr=0.1)
    metadata = load_checkpoint(path, restored, restored_optimizer)
    with torch.inference_mode():
        actual = restored(sample)
    torch.testing.assert_close(actual, expected)
    assert metadata['epoch'] == 1
seed_everything(9); first = torch.randn(3)
seed_everything(9); second = torch.randn(3)
torch.testing.assert_close(first, second)
print('round-trip and seed checks passed')

## 检查点

说明为什么测试参数是否更新比只检查 loss 非空更可靠；解释 checkpoint 测试为什么需要比较输出；指出哪些数值比较应该使用 `torch.testing.assert_close`。

## 试一试

故意漏掉 `optimizer.step()`，观察参数更新测试失败；为 CNN 增加 NCHW 输入和不同图像尺寸的 shape 测试。

## 常见错误与调试

测试依赖训练达到随机准确率、只检查文件存在、使用严格相等比较浮点数、没有覆盖不同 batch size、测试之间共享已更新模型状态。